# Stage 1 — Non-Instruction Fine-Tuning (Healthcare FAQ Assistant)

**Goal:** adapt a base LLM to *healthcare language and terminology* by continuing pre-training on raw domain text **before** instruction tuning.

Pipeline: **Base Model → [Stage 1: Non-Instruction FT] → Stage 2: SFT → Stage 3: DPO**

This notebook:
1. Loads raw domain text (`data/non_instruction_data.txt`)
2. Cleans & chunks it
3. Loads the base model with Unsloth + 4-bit (QLoRA)
4. Applies LoRA (incl. embeddings for continued pre-training)
5. Trains on raw text
6. Saves the adapter + a merged model for Stage 2
7. Tests the model after non-instruction fine-tuning

> ⚠️ Educational project — the assistant gives general health information only and is not medical advice.

## 0. Install dependencies (Colab)

In [ ]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes itself --
# do NOT pin an old trl (e.g. trl<0.12): it passes `tokenizer=` to Trainer.__init__(),
# which newer transformers removed, causing:
#   TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'
#%%capture
!pip install -q unsloth unsloth_zoo
# After installing, restart the runtime once (Runtime -> Restart session) before continuing.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
import unsloth  # Important: import Unsloth early

import time
import torch

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#import torch
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [ ]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

Mounted at /content/drive
REPO_DIR = /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning


## 1. Select base model

In [ ]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

Selected model: qwen2.5-0.5b  ->  unsloth/Qwen2.5-0.5B


## 2. Paths

In [ ]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

Data dir:   /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/data
Output dir: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs


## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = True                 # set True to upload after training
HF_USERNAME  = "your-hf-username"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

Logged in to Hugging Face Hub as mannyiyer


## 3. Load the base model with Unsloth (4-bit / QLoRA)

In [ ]:
import unsloth

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,
    max_seq_length = max_seq_length,
    dtype          = None,        # auto: bf16 on Ampere+, else fp16
    load_in_4bit   = True,        # QLoRA
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print('Base model loaded.')

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded.


In [ ]:
from unsloth import FastLanguageModel
import torch

# Load an untouched copy of the base model
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,          # same base you started from
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(base_model)

prompts = [
    "An employee who has the flu and needs to apply for sick leave should",              # 1
    "In adults, a body temperature is considered a fever when it reaches",               # 2
    "For a common cold, taking antibiotics is",                                          # 3
    "A person taking blood pressure medication who feels fine should",                   # 4
    "If a baby under three months old has a fever, the parents should",                  # 5
    "The warning signs of a stroke include",                                             # 6
    "When a headache will not go away, the amount of paracetamol a person can take is",  # 7
    "A safe and effective way to lose weight is",                                        # 8
    "To manage type 2 diabetes, a person can",                                           # 9
    "This Healthcare FAQ Assistant can help with",                                       # 10
]

gen_kwargs = dict(
    max_new_tokens     = 120,
    do_sample          = True,
    temperature        = 0.7,
    top_p              = 0.9,
    repetition_penalty = 1.15,
)

def complete(mdl, tok, prompt):
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(**inputs, **gen_kwargs)
    return tok.decode(out[0], skip_special_tokens=True)

for p in prompts:
    print("PROMPT:", p)
    print("BASE   :", complete(base_model, base_tokenizer, p))
    print("STAGE 1:", complete(model, tokenizer, p))
    print("-" * 80)

# Optional: free the base model's GPU memory when done
del base_model, base_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: An employee who has the flu and needs to apply for sick leave should


...

BASE   : To manage type 2 diabetes, a person can take a pill every two weeks for four years. The pills are known to be the same as those used by people with type 1 diabetes and cost about $750 each. If you get sick because of it, your insurance covers half of that price.
You must wait until after age fifty if you want to buy another drug like Metformin or Acarbose before you begin to use them in this way. You may also need help finding new ways to treat Type II Diabetes. This is especially true if you have problems such as high blood pressure, kidney disease, an enlarged prostate,


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAGE 1: To manage type 2 diabetes, a person can use the following steps:
1. Eat a healthy diet rich in fruits and vegetables.
2. Exercise regularly for at least 30 minutes every day.
3. Get enough sleep each night to help your body produce insulin properly.
4. Limit your sugar intake from foods like sweets or soda.

In summary: managing type 2 diabetes requires eating well, exercising regularly, getting plenty of rest, and limiting sugars.
--------------------------------------------------------------------------------
PROMPT: This Healthcare FAQ Assistant can help with


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE   : This Healthcare FAQ Assistant can help with a variety of questions regarding health insurance, healthcare coverage options and medical services. Our FAQs are designed to be quick reference guides for patients seeking answers on how to find the best option that meets their needs.
What is covered by my health plan?
How do I know if I am eligible for Medicare Advantage or Health Maintenance Organization plans?
If you have been diagnosed with cancer, what steps should you take next?
STAGE 1: This Healthcare FAQ Assistant can help with any question you may have. If your question is not answered, please contact the Healthcare F.A.Q.'s team.
What is a Health Care Worker?
A Health Care worker (HCW) refers to an individual who has completed a training program in health care and takes part of their duties as healthcare providers within healthcare facilities or hospitals.
Healthcare workers work directly under physicians and nurses; however they also take on some responsibilities such as

## 4. Load, clean and chunk the raw domain text

For continued pre-training we feed plain text. We split the file into paragraphs, drop empties, and append the EOS token so the model learns where passages end.

In [ ]:
import os
from datasets import Dataset

raw_path = os.path.join(DATA_DIR, "non_instruction_data.txt")
with open(raw_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Clean + chunk: one training example per paragraph
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
paragraphs = [" ".join(p.split()) for p in paragraphs]   # collapse whitespace
print(f"Loaded {len(paragraphs)} paragraphs")

EOS = tokenizer.eos_token
texts = [p + EOS for p in paragraphs]
dataset = Dataset.from_dict({"text": texts})
print(dataset)
print("\nExample:\n", dataset[0]["text"][:300])

Loaded 56 paragraphs
Dataset({
    features: ['text'],
    num_rows: 56
})

Example:
 A fever is a temporary rise in body temperature, usually a sign that the body is fighting an infection. For most adults, a temperature at or above 38 degrees Celsius (100.4 degrees Fahrenheit) is considered a fever. Mild fevers are often part of a normal immune response and do not always require med


## 5. Apply LoRA

For continued pre-training we also train the `embed_tokens` and `lm_head` so the model can better absorb new domain vocabulary (this is the Unsloth continued-pretraining recipe).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj",
                      "embed_tokens","lm_head"],   # extra modules for continued pre-training
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)
print('LoRA adapters attached.')

Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
LoRA adapters attached.


## 6. Train on the raw domain text

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model = model,
    processing_class = tokenizer,          # new API: was `tokenizer=` in older trl
    train_dataset = dataset,
    args = UnslothTrainingArguments(
        # Dataset args moved into the config in newer trl (SFTConfig subclass)
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,

        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size 8
        warmup_steps = 5,
        num_train_epochs = 3,              # small dataset -> a couple of passes
        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,    # lower LR for embeddings
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage1_logs"),
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
trainer_stats


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/56 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 56 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 147,378,176 of 777,545,600 (18.95% trained)


Unsloth: Setting lr = 5.00e-06 instead of 5.00e-05 for embed_tokens.


`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.111628
2,1.997245
3,2.076542
4,2.241636
5,2.032532
6,1.939573
7,1.993993
8,1.871176
9,2.227602
10,2.032963


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_logs/checkpoint-21/tokenizer_config.json.


TrainOutput(global_step=21, training_loss=1.9264931735538302, metrics={'train_runtime': 82.1488, 'train_samples_per_second': 2.045, 'train_steps_per_second': 0.256, 'total_flos': 37116585563904.0, 'train_loss': 1.9264931735538302, 'epoch': 3.0})

## 7. Save the adapter and a merged model (input to Stage 2)

In [ ]:
STAGE1_ADAPTER = os.path.join(OUTPUT_DIR, "stage1_non_instruction")
STAGE1_MERGED  = os.path.join(OUTPUT_DIR, "stage1_merged")

# LoRA adapter (small)
model.save_pretrained(STAGE1_ADAPTER)
tokenizer.save_pretrained(STAGE1_ADAPTER)
print("Saved adapter ->", STAGE1_ADAPTER)

# Merged 16-bit model so Stage 2 can load it as a clean base
model.save_pretrained_merged(STAGE1_MERGED, tokenizer, save_method="merged_16bit")
print("Saved merged model ->", STAGE1_MERGED)

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_non_instruction/tokenizer_config.json.


Saved adapter -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_non_instruction


config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:07<00:00, 127.79s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged`
Saved merged model -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged


### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage1"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage1-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage1-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage1-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 8. Test the model after non-instruction fine-tuning

This stage teaches *domain language*, not Q&A behaviour yet — so we test text **completion**. The model should continue healthcare text in a domain-appropriate style.

In [ ]:
FastLanguageModel.for_inference(model)

prompts = [
    "An employee who has the flu and needs to apply for sick leave should",              # 1
    "In adults, a body temperature is considered a fever when it reaches",               # 2
    "For a common cold, taking antibiotics is",                                          # 3
    "A person taking blood pressure medication who feels fine should",                   # 4
    "If a baby under three months old has a fever, the parents should",                  # 5
    "The warning signs of a stroke include",                                             # 6
    "When a headache will not go away, the amount of paracetamol a person can take is",  # 7
    "A safe and effective way to lose weight is",                                        # 8
    "To manage type 2 diabetes, a person can",                                           # 9
    "This Healthcare FAQ Assistant can help with",                                       # 10
]
for p in prompts:
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=120, use_cache=True, do_sample=True, temperature = 0.7,top_p = 0.9,repetition_penalty = 1.15)
    print("PROMPT:", p)
    print("CONT :", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 80)

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

PROMPT: An employee who has the flu and needs to apply for sick leave should
CONT : An employee who has the flu and needs to apply for sick leave should first go through the procedures at ____. 
A. The company's Human Resources Department
B. County-level or above medical institution designated by the provincial, autonomous region, or municipal governments
C. Municipal People's Government Health Administrative Department
D. National Medical Institution
Answer:
ABC

The characteristics of a high-voltage circuit breaker failure protection device include ____.
A. Rapid action time$
B. $Short trip distance$
C. $Quick judgment capability$
D. $Simple structure
Answer:
ABCD

According to the 'Notice on Improving the Service Level Management System' (No
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: In adults, a body temperature is considered a fever when it reaches
CONT : In adults, a body temperature is considered a fever when it reaches or exceeds which of the following?
A. 36.0°C
B. 37.5°C
C. 38.1°C
D. 39.2°C
E. 40.0°C
Answer: D

According to Article 7.2.1 of DL/T476-2010 'Power Quality Technical Supervision Regulations', what does this regulation refer to as power quality? It includes frequency variation and voltage deviation.
A. Voltage
B. Frequency
C. Current
D. Power supply reliability

--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

PROMPT: When a headache will not go away, the amount of paracetamol a person can take is
CONT : When a headache will not go away, the amount of paracetamol a person can take is
A: 100mg. B: 250 mg C: 400 mg D: 750 mg The correct answer is option (B) : 250 mg.

Explanation:
Paracetamol should be taken as directed on the package or within one hour after taking it to avoid side effects such as drowsiness and constipation.
The amount of paracetamol that can be consumed safely in a single day varies from person to person due to factors like age, weight, body mass index (BMI), etc., but generally ranges between
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: A safe and effective way to lose weight is
CONT : A safe and effective way to lose weight is through dieting. You can follow the basic steps in order, which include planning your meals, monitoring your calories intake, and exercise regularly.
Here are some tips for losing weight:
1. Plan Your Meals: Eating healthy food on a regular basis will help you maintain good health while also helping you feel full when eating smaller portions of high-calorie foods like fast-food items or ice cream cones.
2. Monitor Your Caloric Intake: It’s important to keep track of how much food you consume each day so that you don’t overeat during times of stress such as exams or work deadlines;
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: To manage type 2 diabetes, a person can
CONT : To manage type 2 diabetes, a person can take medicines to control blood sugar levels. If the medicine is not enough or too little, it causes the pancreas (a gland in your stomach) to make extra insulin. This happens when there isn’t enough of the hormone glucagon made by the liver.
In people who have no symptoms, sometimes they don't need to be treated with injections for hypoglycemia (low blood glucose), and their doctor may just advise them to eat more food that will help keep their blood sugars from dropping so low.
--------------------------------------------------------------------------------
PROMPT: This Healthcare FAQ Assistant can help with
CONT : This Healthcare FAQ Assistant can help with a wide range of questions about health and wellness. You can ask your question here, or simply send in an email to healthcarefaqs@healthcare.com
What is the difference between chronic disease and non-chronic disease?
Chronic diseases ar

## Done — Stage 1 complete ✅

Next: open **`instruction_finetuning.ipynb`** and set `RESUME_FROM_STAGE1 = True` to continue from `outputs/stage1_merged`.